In [ ]:
# Standard libraries
import numpy as np
from pathlib import Path
from glob import glob
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Specific libraries
import nibabel as nib
from scipy.ndimage import distance_transform_edt

# custom functions
import registration as reg
import temperature as temp

In [ ]:
b0_thresh = 5 
t_ref = 23 # C
adc_ref = temp.water_diffusivity(t_ref)

In [ ]:
parent_dir = Path("./snr")
eval_dir = parent_dir / "derivatives"
plot_dir = parent_dir / "plots"
plot_dir.mkdir(exist_ok=True)
testname = eval_dir.parent.stem

params_fname = eval_dir / (testname + "_" + "params.h5")
scans_fname = eval_dir / (testname + "_" + "RAW_REG.nii.gz")
mask_fname = eval_dir / (testname + "_" + "MASK.nii.gz")

# Load in sequence parameters
snr_df = pd.read_hdf(params_fname, key="df", mode="r")

# Load in scan data and add into data frame
scans_nii = nib.load(scans_fname)
scans_data = scans_nii.get_fdata()
scans_aff = scans_nii.affine
scans_spacing = reg.spacing_from_affine(scans_aff)

data_reshuffled = np.transpose(scans_data,[3,0,1,2]).astype(np.float32)
snr_df["image"] = list(data_reshuffled)

# Load in mask
mask_nii = nib.load(mask_fname)
mask = mask_nii.get_fdata().astype(bool)

In [ ]:
dwi_df = snr_df[snr_df["b_value"] > b0_thresh].copy()
dwi_df["adc"] = dwi_df.apply(lambda row: temp.water_diffusivity(row["temperature"]), axis=1)
adc_ref = dwi_df.iloc[0]["adc"]

dwi_df["scale_factor"] = dwi_df.apply(lambda row: np.exp(-row["b_value"] * (adc_ref - row["adc"])), axis=1)
scaled_images = dwi_df["image"].to_numpy() * dwi_df["scale_factor"].to_numpy()

signal = scaled_images.mean()
noise = scaled_images.std()
snr = signal/noise

plot_data = snr.copy()
plot_data[~mask] = 0
px.imshow(plot_data, animation_frame=2, height=800)

In [ ]:
snr_vec = snr[mask]
n_masked = snr_vec.shape[0]
snr_2 = (np.count_nonzero(snr_vec<=2) / n_masked) * 100
snr_5 = (np.count_nonzero(snr_vec<=5) / n_masked) * 100
snr_avg = np.mean(snr_vec)

print(f"Average SNR: {snr_avg:.4f}")
print(f"Fraction of voxels with SNR < 5: {snr_5:.4f}")
print(f"Fraction of voxels with SNR < 2: {snr_2:.4f}")

fig, ax = plt.subplots(1,1, figsize=(16,9))
hist = sns.kdeplot(snr_vec, clip=(0,50), c="cornflowerblue")

ax.set_xlim(0, 50)
ax.set_title(f"SNR mean: {snr_avg:.2f}, SNR <= 2 (%): {snr_2:.4f}, SNR <= 5 (%): {snr_5:.4f}")
ax.set_xlabel("SNR")

fig.tight_layout()
fig.show()